# Historical_AI on Colab

Runtime > Change runtime type > **T4 GPU** before running.

Steps: check GPU -> install deps -> install/start Ollama (2 models) -> HF login -> pull project from Drive -> install the local `vectordb` package -> run `main.py`.

In [ ]:
!nvidia-smi

In [ ]:
%pip install -q transformers==5.13.0 peft bitsandbytes==0.49.2 accelerate==1.14.0 \
    langchain langchain-core langchain-community langchain-ollama langchain-chroma \
    chromadb rich
# Pinned to versions confirmed working on the dev box — newer/older transformers or
# bitsandbytes have caused cache/quantization bugs on this model stack before.

## Ollama (two models: chat orchestrator + embeddings)
`gemma4:e4b` runs the judge/debrief/orchestrator calls. `embeddinggemma:latest` is what `vectordb/vector.py` uses to embed and search the RAG documents — miss this one and vector search fails silently. Re-run this cell if the runtime restarts.

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time
subprocess.Popen(["ollama", "serve"])
time.sleep(3)
!ollama pull gemma4:e4b
!ollama pull embeddinggemma:latest

## Hugging Face login
`google/gemma-4-E2B-it` is gated — log in with a token whose account accepted the model license.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## Project files
Zip the whole `Historical_AI` folder (not just `main/` — it reaches into sibling `run/`, `rag/`, `train/` dirs) and drop it in your Drive as `Historical_AI.zip`, then run this cell.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!unzip -q -o '/content/drive/MyDrive/Historical_AI.zip' -d /content/
%cd /content/Historical_AI/main

## Make `vectordb` importable
`main.py` does `from vectordb.vector import HistoricalVector`. On your machine this only resolves because `rag/infastructure` (package name `historical`) is `pip install -e`'d into your conda env — a plain zip+cd doesn't give you that. Install it the same way here.

In [ ]:
%pip install -q -e /content/Historical_AI/rag/infastructure

In [ ]:
!python main.py